In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import os

# Assuming 'path' variable from the previous cell contains the directory to the dataset files
data_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(data_path)


In [ ]:
display(df.head())


In [ ]:
df.info()

In [ ]:
display(df.describe())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(df['Delivery_Time'].dropna(), kde=True)
plt.title('Distribution of Delivery Time')
plt.xlabel('Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

In [ ]:
# Task 1: Drop the 'Order_ID' column (already dropped in previous execution, so this line is commented out or removed if re-executing this cell from scratch)
# df = df.drop('Order_ID', axis=1)

# Task 2: Handle missing values appropriately
# Checking for missing values
print('Missing values before handling:')
print(df.isnull().sum())

# For numerical columns, fill with median
# 'Courier_Experience_yrs' and 'Delivery_Time' are numerical
df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].median())
df['Delivery_Time'] = df['Delivery_Time'].fillna(df['Delivery_Time'].median())

# For categorical columns, fill with mode
# 'Weather', 'Traffic_Level', 'Time_of_Day' are categorical
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df[col] = df[col].fillna(df[col].mode()[0])

print('\nMissing values after handling:')
print(df.isnull().sum())

In [ ]:
# Task 3: Check and remove duplicates
initial_rows = df.shape[0]
df.drop_duplicates(inplace=True)
final_rows = df.shape[0]

print(f"Number of rows before removing duplicates: {initial_rows}")
print(f"Number of rows after removing duplicates: {final_rows}")
print(f"Number of duplicate rows removed: {initial_rows - final_rows}")

In [ ]:
# Task 4: Encode categorical variables
df_encoded = pd.get_dummies(df, columns=['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type'], drop_first=True)

df = df_encoded.copy()

print('DataFrame after one-hot encoding:')
display(df.head())

In [ ]:
from sklearn.preprocessing import StandardScaler

# Separate features (X) and target (y)
X = df.drop('Delivery_Time', axis=1)
y = df['Delivery_Time']

# Initialize StandardScaler
scaler = StandardScaler()

# Apply scaling to features
X_scaled = scaler.fit_transform(X)

# Convert scaled features back to DataFrame for better readability and to maintain column names
X = pd.DataFrame(X_scaled, columns=X.columns)

print('Features after scaling:')
display(X.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(y, kde=True)
plt.title('Distribution of Delivery Time (Target)')
plt.xlabel('Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

print(f"Skewness of Delivery_Time: {y.skew():.2f}")

if abs(y.skew()) > 0.5:
    print("The target variable 'Delivery_Time' shows significant skewness (imbalance).")
else:
    print("The target variable 'Delivery_Time' does not show significant skewness (is balanced).")

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Split the dataset into features (X) and target (y)
# X and y are already defined from the scaling step
# X = df.drop('Delivery_Time', axis=1)
# y = df['Delivery_Time']

print("Dataset split into features (X) and target (y) has been performed in the previous scaling step.")

In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

# Task 2: Use KFold for splitting
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Task 3: Train a RandomForest model
    model = RandomForestRegressor(random_state=42)
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)

    # Task 4: Evaluate using MAE
    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)

# Task 5: Print the averaged score across all folds
print(f"Average MAE across all folds: {np.mean(mae_scores):.2f}")
print(f"Standard deviation of MAE across all folds: {np.std(mae_scores):.2f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Get feature importances from the trained model
feature_importances = model.feature_importances_

# Create a DataFrame for better visualization
features_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(12, 7))
sns.barplot(x='Importance', y='Feature', data=features_df)
plt.title('Feature Importance from RandomForest Model')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.grid(axis='x', linestyle='--')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Assuming 'model' is the last trained model and 'X' is the scaled features DataFrame
# We need to get predictions for the entire dataset or a representative sample.
# For consistency with the MAE calculation, let's re-run predictions on the full dataset using the trained model

# The 'model' variable holds the last trained RandomForestRegressor from the KFold loop.
# To get predictions for the entire dataset, we can retrain one model on the full X and y,
# or simply use the last 'y_pred' if it was representative.
# However, 'y_pred' from the KFold loop only contains predictions for the last test fold.
# For a full distribution, it's better to train a final model on the entire dataset.

final_model = RandomForestRegressor(random_state=42)
final_model.fit(X, y)

all_predictions = final_model.predict(X)

plt.figure(figsize=(10, 6))
sns.histplot(all_predictions, kde=True)
plt.title('Distribution of Predicted Delivery Time')
plt.xlabel('Predicted Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

In [ ]:
!pip install catboost

from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

# Initialize KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

ensemble_mae_scores = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Initialize and train RandomForestRegressor
    rf_model = RandomForestRegressor(random_state=42, n_estimators=100, n_jobs=-1)
    rf_model.fit(X_train, y_train)
    rf_predictions = rf_model.predict(X_test)

    # Initialize and train CatBoostRegressor
    # verbose=0 to suppress extensive CatBoost output during training
    cat_model = CatBoostRegressor(random_state=42, verbose=0, iterations=100, early_stopping_rounds=10)
    cat_model.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=10)
    cat_predictions = cat_model.predict(X_test)

    # Average their predictions
    avg_predictions = (rf_predictions + cat_predictions) / 2

    # Calculate MAE on the averaged predictions
    mae = mean_absolute_error(y_test, avg_predictions)
    ensemble_mae_scores.append(mae)

# Print the averaged MAE across all folds for the ensemble
print(f"Average Ensemble MAE across all folds: {np.mean(ensemble_mae_scores):.2f}")
print(f"Standard deviation of Ensemble MAE across all folds: {np.std(ensemble_mae_scores):.2f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 8))
sns.scatterplot(x=y, y=all_predictions, alpha=0.6)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2, label='Perfect Prediction')
plt.title('Predicted vs. Actual Delivery Times')
plt.xlabel('Actual Delivery Time (minutes)')
plt.ylabel('Predicted Delivery Time (minutes)')
plt.grid(True)
plt.legend()
plt.show()